In [ ]:
%sql
/* ===================================================================================
   EBA04 - Preview newly added rows in the combined table
   Rows in ritm16070584_combined_04272026 that do NOT exist in the original 7 files.
   Expensecategory is excluded from the comparison (formatting differs).
=================================================================================== */

WITH Old_Coupa_Union AS (
    SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
)

SELECT c.DateExpenseIncurrred, c.PublicOfficial, c.Expensecategory, c.Total,
       c.BusinessPurpose, c.Account, c.Divisor
FROM hive_metastore.ra_adido_2025.ritm16070584_combined_04272026 c
WHERE NOT EXISTS (
    SELECT 1 FROM Old_Coupa_Union o
    WHERE TRIM(c.DateExpenseIncurrred) <=> TRIM(o.DateExpenseIncurrred)
      AND TRIM(c.PublicOfficial) <=> TRIM(o.PublicOfficial)
      AND TRIM(c.Total) <=> TRIM(o.Total)
      AND TRIM(c.BusinessPurpose) <=> TRIM(o.BusinessPurpose)
      AND TRIM(c.Account) <=> TRIM(o.Account)
)
ORDER BY c.DateExpenseIncurrred, c.Account

In [ ]:
# ===================================================================================
# EXPORT 1: Newly added rows -> CSV
# EXPORT 2: Reduced combined table (new rows removed) -> CSV
# ===================================================================================
from pyspark.sql.functions import col, trim
from functools import reduce
from operator import and_

def safe_rm(path):
    try:
        dbutils.fs.rm(path, recurse=True)
    except Exception:
        pass

def export_single_csv(df, output_dir, final_path):
    safe_rm(output_dir)
    safe_rm(final_path)
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_dir)
    part_file = [f.path for f in dbutils.fs.ls(output_dir) if f.name.startswith("part-")][0]
    dbutils.fs.cp(part_file, final_path)
    safe_rm(output_dir)
    return final_path

old_union_tables = [
    "hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered",
]

# Columns for matching (excluding Expensecategory due to format drift)
match_cols = ["DateExpenseIncurrred", "PublicOfficial", "Total", "BusinessPurpose", "Account"]

combined_table = "hive_metastore.ra_adido_2025.ritm16070584_combined_04272026"
df_combined = spark.table(combined_table)

# Build old union with TRIM-normalized match columns
df_old = None
for t in old_union_tables:
    df_t = spark.table(t).select(*[trim(col(c)).alias(c) for c in match_cols])
    df_old = df_t if df_old is None else df_old.unionAll(df_t)

# Build join condition
join_cond = [trim(df_combined[c]).eqNullSafe(df_old[c]) for c in match_cols]

# --- EXPORT 1: Newly added rows (NOT in old files) ---
df_new_rows = df_combined.join(df_old, reduce(and_, join_cond), "left_anti")
df_new_rows = df_new_rows.orderBy("DateExpenseIncurrred", "Account")

new_count = df_new_rows.count()
print(f"New rows found: {new_count}")
export_single_csv(df_new_rows, "dbfs:/tmp/eba04_new_rows_staging", "dbfs:/tmp/eba04_new_rows.csv")
print("Exported: dbfs:/tmp/eba04_new_rows.csv")

# --- EXPORT 2: Reduced combined table (new rows removed) ---
df_reduced = df_combined.join(df_old, reduce(and_, join_cond), "left_semi")
df_reduced = df_reduced.orderBy("DateExpenseIncurrred", "Account")

reduced_count = df_reduced.count()
print(f"Reduced table rows: {reduced_count}")
export_single_csv(df_reduced, "dbfs:/tmp/eba04_reduced_staging", "dbfs:/tmp/eba04_reduced_combined.csv")
print("Exported: dbfs:/tmp/eba04_reduced_combined.csv")

print(f"\nSummary:")
print(f"  Combined table total:  {df_combined.count()}")
print(f"  New rows (exported):   {new_count}")
print(f"  Reduced rows (exported): {reduced_count}")
print(f"  New + Reduced:         {new_count + reduced_count}")